# WACC Deep Dive

Weighted Average Cost of Capital is one of the most important assumptions in valuation. This notebook goes deeper than the foundations notebook by estimating WACC from components, testing scenarios, and measuring valuation impact.

Abbreviations used in this notebook:

- **WACC**: Weighted Average Cost of Capital, the blended required return of debt and equity investors.
- **CAPM**: Capital Asset Pricing Model, a method for estimating cost of equity.
- **ERP**: Equity Risk Premium, the extra return investors demand for equities over a risk-free asset.
- **COD**: Cost of Debt, the borrowing rate a company pays before tax effects.
- **EV**: Enterprise Value, the value of the operating business.
- **FCF**: Free Cash Flow, cash generated after capital expenditures.
- **DCF**: Discounted Cash Flow, a valuation method based on present value of future cash flows.
- **CHF**: Swiss franc, the currency used in the examples.

## 1. Intuition

WACC is not a number we simply look up. It is an estimate built from assumptions about market risk, company risk, borrowing cost, tax effects, and capital structure.

A lower WACC usually means investors see the business as safer or capital is cheaper. A higher WACC usually means higher risk, higher interest rates, weaker credit quality, or more uncertainty.

In valuation, WACC matters because it controls how aggressively future free cash flows are discounted. A small WACC change can create a large valuation change, especially when terminal value is large.

## 2. Mathematics

Cost of equity using CAPM:

$$
r_e = r_f + \beta \times ERP
$$

Where:

- $r_e$ = cost of equity
- $r_f$ = risk-free rate
- $\beta$ = beta, sensitivity to market risk
- $ERP$ = equity risk premium

After-tax cost of debt:

$$
r_d^{after-tax} = r_d \times (1 - T_c)
$$

WACC:

$$
WACC = w_e r_e + w_d r_d(1 - T_c)
$$

Where:

- $w_e = \frac{E}{D + E}$ = equity weight
- $w_d = \frac{D}{D + E}$ = debt weight
- $E$ = market value of equity
- $D$ = market value of debt
- $T_c$ = corporate tax rate

Enterprise value from a simple growing perpetuity:

$$
EV = \frac{FCF_1}{WACC - g}
$$

This simplified formula shows why valuation is highly sensitive when $WACC$ approaches long-term growth $g$.

## 3. Implementation

We will estimate WACC for a mature consumer staples company, then build downside, base, and upside scenarios. The numbers are synthetic but intentionally realistic enough for valuation practice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")


def cost_of_equity(risk_free_rate, beta, equity_risk_premium):
    return risk_free_rate + beta * equity_risk_premium


def after_tax_cost_of_debt(pre_tax_cost_of_debt, tax_rate):
    return pre_tax_cost_of_debt * (1 - tax_rate)


def wacc(equity_value, debt_value, cost_equity, pre_tax_cost_of_debt, tax_rate):
    total_capital = equity_value + debt_value
    equity_weight = equity_value / total_capital
    debt_weight = debt_value / total_capital
    return equity_weight * cost_equity + debt_weight * after_tax_cost_of_debt(pre_tax_cost_of_debt, tax_rate)


def enterprise_value_growing_perpetuity(next_year_fcf, discount_rate, long_term_growth):
    if discount_rate <= long_term_growth:
        raise ValueError("Discount rate must be greater than long-term growth.")
    return next_year_fcf / (discount_rate - long_term_growth)

In [ ]:
scenarios = pd.DataFrame({
    "scenario": ["Downside", "Base", "Upside"],
    "risk_free_rate": [0.020, 0.015, 0.012],
    "beta": [0.85, 0.70, 0.60],
    "equity_risk_premium": [0.060, 0.055, 0.050],
    "pre_tax_cost_of_debt": [0.045, 0.032, 0.026],
    "tax_rate": [0.21, 0.21, 0.21],
    "equity_value": [230_000, 250_000, 270_000],
    "debt_value": [55_000, 45_000, 38_000],
    "next_year_fcf": [10_200, 10_800, 11_400],
    "long_term_growth": [0.015, 0.020, 0.025],
})

scenarios["cost_of_equity"] = scenarios.apply(
    lambda row: cost_of_equity(row["risk_free_rate"], row["beta"], row["equity_risk_premium"]),
    axis=1,
)
scenarios["after_tax_cost_of_debt"] = scenarios.apply(
    lambda row: after_tax_cost_of_debt(row["pre_tax_cost_of_debt"], row["tax_rate"]),
    axis=1,
)
scenarios["equity_weight"] = scenarios["equity_value"] / (scenarios["equity_value"] + scenarios["debt_value"])
scenarios["debt_weight"] = scenarios["debt_value"] / (scenarios["equity_value"] + scenarios["debt_value"])
scenarios["wacc"] = scenarios.apply(
    lambda row: wacc(row["equity_value"], row["debt_value"], row["cost_of_equity"], row["pre_tax_cost_of_debt"], row["tax_rate"]),
    axis=1,
)
scenarios["enterprise_value"] = scenarios.apply(
    lambda row: enterprise_value_growing_perpetuity(row["next_year_fcf"], row["wacc"], row["long_term_growth"]),
    axis=1,
)

scenarios[[
    "scenario", "cost_of_equity", "after_tax_cost_of_debt", "equity_weight", "debt_weight", "wacc", "enterprise_value"
]].round(4)

A WACC build should be decomposed. If the final WACC looks surprising, the cause is usually visible in beta, the equity risk premium, cost of debt, or capital structure weights.

In [ ]:
base = scenarios.loc[scenarios["scenario"] == "Base"].iloc[0]

wacc_bridge = pd.Series({
    "Risk-free rate": base["risk_free_rate"],
    "Beta x ERP": base["beta"] * base["equity_risk_premium"],
    "Cost of equity": base["cost_of_equity"],
    "After-tax cost of debt": base["after_tax_cost_of_debt"],
    "Equity weight": base["equity_weight"],
    "Debt weight": base["debt_weight"],
    "WACC": base["wacc"],
})

wacc_bridge.to_frame("base_case")

## 4. Visualization

Good WACC work is scenario-based. The charts below show how the cost of equity, debt, capital structure, and final WACC differ across cases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

scenarios.set_index("scenario")[["cost_of_equity", "after_tax_cost_of_debt", "wacc"]].plot(
    kind="bar",
    ax=axes[0],
    color=["#2f6f8f", "#9a6b2f", "#4f7f45"],
)
axes[0].set_title("Required Return by Scenario")
axes[0].set_xlabel("")
axes[0].set_ylabel("Rate")
axes[0].yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
axes[0].tick_params(axis="x", rotation=0)

scenarios.set_index("scenario")[["equity_weight", "debt_weight"]].plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=["#2f6f8f", "#9a6b2f"],
)
axes[1].set_title("Capital Structure Weights")
axes[1].set_xlabel("")
axes[1].set_ylabel("Weight")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
wacc_range = np.linspace(0.045, 0.085, 9)
growth_range = np.linspace(0.010, 0.030, 9)

sensitivity = pd.DataFrame(index=wacc_range, columns=growth_range, dtype=float)
for discount_rate in wacc_range:
    for growth in growth_range:
        sensitivity.loc[discount_rate, growth] = enterprise_value_growing_perpetuity(
            next_year_fcf=base["next_year_fcf"],
            discount_rate=discount_rate,
            long_term_growth=growth,
        )

sensitivity.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
image = ax.imshow(sensitivity.values, aspect="auto", cmap="viridis")

ax.set_xticks(range(len(growth_range)))
ax.set_xticklabels([f"{g:.1%}" for g in growth_range])
ax.set_yticks(range(len(wacc_range)))
ax.set_yticklabels([f"{r:.1%}" for r in wacc_range])
ax.set_xlabel("Long-term FCF growth")
ax.set_ylabel("WACC")
ax.set_title("Enterprise Value Sensitivity: WACC vs Growth")

for row in range(sensitivity.shape[0]):
    for col in range(sensitivity.shape[1]):
        ax.text(col, row, f"{sensitivity.values[row, col] / 1_000:.0f}", ha="center", va="center", color="white", fontsize=8)

fig.colorbar(image, ax=ax, label="Enterprise value, CHF millions")
plt.tight_layout()
plt.show()

## 5. Application

In a real valuation, WACC should be estimated with market-consistent inputs:

- Use a risk-free rate in the same currency as the cash flows.
- Use a beta that reflects the company’s long-term business risk, not just a noisy short period.
- Use a defensible equity risk premium for the market and region.
- Estimate cost of debt from current borrowing rates or bond yields, not old coupon rates.
- Use market-value capital structure where possible.
- Compare current capital structure with a sustainable target capital structure.

For a stable consumer staples company, the WACC range is often narrower than for a cyclical or highly leveraged company. Still, even a narrow range can move valuation meaningfully.

In [ ]:
low_ev = sensitivity.min().min()
high_ev = sensitivity.max().max()
base_ev = base["enterprise_value"]

print(f"Base WACC: {base['wacc']:.2%}")
print(f"Base enterprise value: CHF {base_ev:,.0f}m")
print(f"Sensitivity range: CHF {low_ev:,.0f}m to CHF {high_ev:,.0f}m")
print(f"High / low valuation spread: {high_ev / low_ev:.1f}x")

## 6. Reflection

- WACC is an estimate, not an observable fact.
- Cost of equity is often the largest and most judgment-heavy input.
- Cost of debt should reflect current credit conditions and tax effects.
- Market-value capital structure is usually more relevant than book-value capital structure.
- WACC and long-term growth must be internally consistent.
- A valuation should use a WACC range, not just one precise-looking number.

Questions to answer after running the notebook:

1. Which input changes WACC the most in the scenarios?
2. Why should the risk-free rate match the currency of cash flows?
3. What happens when long-term growth gets close to WACC?
4. Which WACC input would you research most carefully before valuing a real company?